# Build your own AI from scratch: a tiny story-writing language model
**SkillWisor** · for software engineers who are new to AI · free GPU (Colab T4 or Kaggle)

We go from raw text to a model you can share, in 10 steps. Every line has a comment, so you can read the code like a story.
**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run each cell from top to bottom.

Quick vocabulary for software engineers:
- **Tensor** = a multi-dimensional array (like a NumPy array) that can live on the GPU.
- **Model** = a function with millions of adjustable numbers (parameters) inside it.
- **Training** = a loop that calls the function, measures how wrong it was, and nudges the numbers to be less wrong.

## Step 1 · Your free AI lab: check the GPU

In [ ]:
# Ask the machine which GPU we got and how much memory it has
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Install two libraries: "datasets" downloads training data, "tokenizers" builds our vocabulary
!pip -q install datasets tokenizers
# PyTorch is the library we use to build and train the model (already installed on Colab)
import torch
# Use the GPU ("cuda") if there is one, otherwise fall back to the slower CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
# Print where training will run
print("Training on:", device)

## Step 2 · Get the data
TinyStories: about 2 million short stories written with words a 4-year-old knows.
On Kaggle instead? Click "+ Add Input", pick a dataset, and read its files from /kaggle/input/.

In [ ]:
# Hugging Face "datasets" can download thousands of public datasets with one line
from datasets import load_dataset
# Download TinyStories (it comes split into "train" and "validation")
ds = load_dataset("roneneldan/TinyStories")
# Show how many stories are in each split
print(ds)
# Print the first 300 characters of the first story, so we see what the data looks like
print(ds["train"][0]["text"][:300])

## Step 3 · Clean the data

In [ ]:
# hashlib turns any text into a short "fingerprint", which we use to spot duplicates
import hashlib

# A function that takes a list of stories and returns a cleaned list
def clean(texts, min_words=20):
    # "seen" remembers fingerprints we already kept, "out" collects the good stories
    seen, out = set(), []
    # Look at every story one by one
    for t in texts:
        # Collapse messy spacing (tabs, double spaces, new lines) into single spaces
        t = " ".join(t.split())
        # Skip stories that are too short to teach anything (often broken rows)
        if len(t.split()) < min_words: continue
        # Make a fingerprint of the lower-cased story
        h = hashlib.md5(t.lower().encode()).hexdigest()
        # If we've seen this fingerprint before, it's a duplicate, so skip it
        if h in seen: continue
        # Otherwise remember the fingerprint and keep the story
        seen.add(h); out.append(t)
    # Hand back the cleaned list
    return out

# Use a slice of 300,000 stories, which trains in minutes on a free GPU
N_TRAIN = 300_000
# Clean the training stories (what the model learns from)
train_txt = clean(ds["train"]["text"][:N_TRAIN])
# Clean 5,000 validation stories (held back to test the model later, never trained on)
val_txt   = clean(ds["validation"]["text"][:5_000])
# Show how many stories survived cleaning
print(f"train: {len(train_txt):,} stories | val: {len(val_txt):,}")

## Step 4 · Train your own tokenizer (BPE)
Models can't read text, only numbers. A tokenizer cuts text into pieces (tokens) and gives each piece an ID.

In [ ]:
# Import the building blocks of a tokenizer
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
# Use Byte Pair Encoding (BPE): start from single characters, then merge the most frequent pairs
tok = Tokenizer(models.BPE(unk_token="<unk>"))
# Work on raw bytes, so any character (even emoji) can be represented
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
# The matching decoder turns token IDs back into normal text
tok.decoder = decoders.ByteLevel()
# Learn 4,096 tokens, plus two special ones: <unk> (unknown) and <eos> (end of story)
trainer = trainers.BpeTrainer(vocab_size=4096, special_tokens=["<unk>", "<eos>"],
                              initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
# Train the tokenizer on our cleaned stories (it counts pairs and merges them)
tok.train_from_iterator(train_txt, trainer)
# Remember the ID of the end-of-story token, we use it to separate stories
EOS = tok.token_to_id("<eos>")
# Try it on a sentence
enc = tok.encode("Once upon a time, a little dog found a red ball.")
# Show how many tokens it became and what they are ("Ġ" just means "there was a space before")
print(len(enc.ids), "tokens:", enc.tokens)

In [ ]:
# NumPy helps us store millions of IDs compactly
import numpy as np
# Turn a list of stories into one long tensor of token IDs
def to_ids(texts):
    # Encode all stories in fast batches, adding <eos> after each one (16-bit ints save memory)
    chunks = [np.array(e.ids + [EOS], dtype=np.int16) for e in tok.encode_batch(texts)]
    # Glue every story into one long array of IDs and turn it into a PyTorch tensor
    return torch.from_numpy(np.concatenate(chunks).astype(np.int64))
# Our training data and validation data, now as numbers
train_ids, val_ids = to_ids(train_txt), to_ids(val_txt)
# How many tokens the model will learn from
print(f"{len(train_ids):,} training tokens")

## Step 5 · Build the brain: a tiny GPT
A GPT is a stack of identical "blocks". Each block = attention (look back at earlier words) + a small neural network (think about them).

In [ ]:
# nn has the neural-network layers, F has helper maths functions
import torch.nn as nn, torch.nn.functional as F

# One transformer block (we will stack six of them)
class Block(nn.Module):
    # d = size of each token's vector, h = number of attention heads
    def __init__(s, d, h):
        # Required setup for every PyTorch module
        super().__init__()
        # Two normalisation layers keep the numbers in a healthy range
        s.ln1, s.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        # Attention: each token looks at earlier tokens and decides which ones matter
        s.attn = nn.MultiheadAttention(d, h, batch_first=True)
        # A small feed-forward network: expand to 4x, apply a non-linearity, shrink back
        s.mlp = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
    # What happens when data flows through the block
    def forward(s, x, mask):
        # Normalise, then let tokens look at each other (the mask hides the future)
        a = s.ln1(x)
        # Add what attention found back onto the input (a "residual" connection)
        x = x + s.attn(a, a, a, attn_mask=mask, need_weights=False)[0]
        # Normalise again, think with the MLP, and add that too
        return x + s.mlp(s.ln2(x))

# The full model
class TinyGPT(nn.Module):
    # Default size: 4,096-token vocab, 384-number vectors, 6 blocks, 6 heads, 256-token context
    def __init__(s, vocab=4096, d=384, layers=6, heads=6, ctx=256):
        # Required setup for every PyTorch module
        super().__init__()
        # Remember the longest text the model can look at
        s.ctx = ctx
        # Embeddings: a lookup table that turns each token ID and each position into a vector
        s.tok, s.pos = nn.Embedding(vocab, d), nn.Embedding(ctx, d)
        # Stack the transformer blocks
        s.blocks = nn.ModuleList([Block(d, heads) for _ in range(layers)])
        # Final normalisation and the "head" that scores every token as the possible next one
        s.ln, s.head = nn.LayerNorm(d), nn.Linear(d, vocab, bias=False)
        # Weight tying: reading and writing tokens share the same table (saves ~1.6M parameters)
        s.head.weight = s.tok.weight
        # Start every weight as a small random number, so the model begins knowing nothing
        for m in s.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, std=0.02)
    # Run the model on a batch of token IDs
    def forward(s, idx):
        # T = how many tokens are in each sequence
        T = idx.shape[1]
        # Token meaning + position in the sentence
        x = s.tok(idx) + s.pos(torch.arange(T, device=idx.device))
        # Causal mask: True above the diagonal means "you can't look at future tokens"
        mask = torch.triu(torch.ones(T, T, device=idx.device, dtype=torch.bool), 1)
        # Pass through every block in order
        for b in s.blocks: x = b(x, mask)
        # Score all 4,096 tokens at every position
        return s.head(s.ln(x))

# Create the model and move it onto the GPU
model = TinyGPT().to(device)
# Count every adjustable number in the model
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

## Step 6 · Pretrain: learn to predict the next token

In [ ]:
# math for exp(), time to measure how long training takes
import math, time
# Batch size, context length, number of training steps, learning rate (how big each nudge is)
B, CTX, STEPS, LR = 64, 256, 3000, 6e-4

# Pick B random chunks of text; y is the same chunk shifted by one token (the "next word" answers)
def batch(data):
    # Random starting positions
    i = torch.randint(len(data) - CTX - 1, (B,))
    # Inputs (x) and targets (y = x shifted one token to the right)
    x = torch.stack([data[j:j+CTX] for j in i]); y = torch.stack([data[j+1:j+CTX+1] for j in i])
    # Move both onto the GPU
    return x.to(device), y.to(device)

# Measure the loss on validation data (no learning happens here)
@torch.no_grad()
def val_loss(n=20):
    # Switch to evaluation mode, then average the loss over n random validation batches
    model.eval(); L = [F.cross_entropy(model(x).flatten(0,1), y.flatten()).item() for x, y in (batch(val_ids) for _ in range(n))]
    # Switch back to training mode and return the average
    model.train(); return sum(L)/n

# AdamW = the optimizer that decides how to nudge each parameter
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)
# Learning-rate schedule: warm up for the first 5% of steps, then slowly cool down
sched = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=STEPS, pct_start=0.05)
# Mixed precision: do most maths in fast 16-bit numbers, the scaler keeps it stable
scaler = torch.amp.GradScaler(enabled=(device == "cuda"))
# Start the clock and a list to record progress
t0, history = time.time(), []
# The training loop
for step in range(STEPS + 1):
    # 1. Get a batch of text and the correct next tokens
    x, y = batch(train_ids)
    # 2. Forward pass in 16-bit: predict, and measure how wrong we were (cross-entropy loss)
    with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
        loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
    # 3. Backward pass: work out which direction to nudge every parameter
    opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
    # 4. Clip very big nudges, then update the parameters
    nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
    # 5. Move the learning-rate schedule forward
    if step < STEPS: sched.step()
    # Every 250 steps, report training loss, validation loss and time so far
    if step % 250 == 0:
        vl = val_loss(); history.append((step, loss.item(), vl))
        print(f"step {step:5d} | train {loss.item():.2f} | val {vl:.2f} | {time.time()-t0:.0f}s")
# Save the trained weights (a checkpoint), so a disconnect never loses the work
torch.save(model.state_dict(), "tiny_gpt_pretrained.pt")

## Step 7 · Talk to it

In [ ]:
# No learning while generating
@torch.no_grad()
# temperature = creativity, top_k = only choose from the k most likely tokens
def generate(prompt, n=120, temperature=0.8, top_k=40):
    # Evaluation mode
    model.eval()
    # Turn the prompt into token IDs, shaped as a batch of one
    idx = torch.tensor([tok.encode(prompt).ids], device=device)
    # Add up to n new tokens, one at a time
    for _ in range(n):
        # Score every possible next token (only the last position matters), divided by temperature
        logits = model(idx[:, -model.ctx:])[:, -1] / temperature
        # Keep the top_k scores and hide all the rest
        v, _ = torch.topk(logits, top_k); logits[logits < v[:, [-1]]] = -float("inf")
        # Turn scores into probabilities and pick one token at random by those odds
        nxt = torch.multinomial(F.softmax(logits, -1), 1)
        # Stop if the model says the story is over
        if nxt.item() == EOS: break
        # Append the new token and go again
        idx = torch.cat([idx, nxt], 1)
    # Turn the IDs back into text
    return tok.decode(idx[0].tolist())

# Ask our model to continue a story
print(generate("Once upon a time, a little robot"))

## Step 8 · Teach it to follow instructions (SFT = supervised fine-tuning)

In [ ]:
# random helps us pick topics and shuffle examples
import random
# Ten topics we will ask for
WORDS = ["dog", "cat", "ball", "tree", "rain", "cake", "boat", "bird", "car", "flower"]

# Build an (instruction, story) pair from a story whose FIRST sentence names one of our topics
def make_pair(story):
    # The first sentence usually introduces what the story is about
    first = story.split(".")[0].lower()
    # Which topic words appear in that first sentence?
    w = [k for k in WORDS if f" {k}" in first]
    # No topic found, so we can't use this story
    if not w: return None
    # The instruction asks for that topic, and the answer is the story itself
    return f"Instruction: Write a short story about a {w[0]}.\nStory: ", story

# Make pairs from all our training stories, keeping only the ones that worked
pairs = [p for p in map(make_pair, train_txt) if p]

# A test: ask for a story about a topic, and check if the topic shows up in the answer
def follows(word, tries=5):
    # Count how many of the tries mention the requested word
    hits = sum(word in generate(f"Instruction: Write a short story about a {word}.\nStory: ", 80).split("Story:")[1].lower() for _ in range(tries))
    # Return the success rate
    return hits / tries

# Score the model BEFORE fine-tuning, so we can compare later
print(f"{len(pairs):,} pairs | before SFT, follows the instruction: {sum(follows(w) for w in WORDS)/len(WORDS):.0%}")

# Build a training batch of instruction + story examples
def sft_batch(bs=32):
    # Lists for inputs and labels
    xs, ys = [], []
    # Pick random pairs
    for prompt, story in random.sample(pairs, bs):
        # Tokenize the instruction and the story (+ end token)
        p, s = tok.encode(prompt).ids, tok.encode(story).ids + [EOS]
        # Join them and cut to the context length
        ids = (p + s)[:CTX + 1]
        # Labels: -100 means "ignore", so the model only learns from the story part
        lab = ([-100] * len(p) + s)[:CTX + 1]
        # Pad short examples to the same length
        pad = CTX + 1 - len(ids)
        xs.append(ids + [EOS] * pad); ys.append(lab + [-100] * pad)
    # Convert to tensors
    x, y = torch.tensor(xs), torch.tensor(ys)
    # Inputs are everything but the last token, labels are shifted by one (next-token again)
    return x[:, :-1].to(device), y[:, 1:].to(device)

# A gentler learning rate (half of pretraining), so we add a habit without erasing knowledge
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
# 1,500 fine-tuning steps
for step in range(1501):
    # Get a batch of instruction examples
    x, y = sft_batch()
    # Forward pass in 16-bit; ignore_index skips the -100 labels
    with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
        loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten(), ignore_index=-100)
    # Backward pass and update, same as pretraining
    opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    # Report progress every 500 steps
    if step % 500 == 0: print(f"sft step {step} | loss {loss.item():.2f}")
# Try an instruction
print(generate("Instruction: Write a short story about a boat.\nStory: "))

## Step 9 · Test it

In [ ]:
# Validation loss and perplexity (roughly: how many tokens the model is torn between at each step)
print(f"val loss {val_loss():.2f} | perplexity {math.exp(val_loss()):.1f}")
# The instruction test again, now AFTER fine-tuning
score = sum(follows(w) for w in WORDS) / len(WORDS)
# Compare with the "before" number from Step 8
print(f"after SFT, follows the instruction: {score:.0%}")

## Step 10 · Shrink it and share it

In [ ]:
# os lets us check file sizes
import os
# Copy all weights to the CPU as 32-bit numbers
sd = {k: v.float().cpu() for k, v in model.state_dict().items()}
# Save the full-size model and the tokenizer
torch.save(sd, "tiny_gpt_fp32.pt"); tok.save("tokenizer.json")

# 8-bit quantization: store each weight as a whole number from -127 to 127, plus one scale per table
def quantize(w):
    # The scale maps the biggest weight to 127
    scale = w.abs().max() / 127
    # Divide, round, and store as 8-bit integers
    return (w / scale).round().to(torch.int8), scale

# Quantize every 2-D weight table (the big ones); keep small 1-D vectors as they are
q = {k: quantize(v) if v.ndim == 2 else v for k, v in sd.items()}
# Save the small version
torch.save(q, "tiny_gpt_int8.pt")
# Compare file sizes
for f in ["tiny_gpt_fp32.pt", "tiny_gpt_int8.pt"]: print(f, f"{os.path.getsize(f)/1e6:.1f} MB")

# Build a fresh model and load the 8-bit weights back (integer x scale = approximate original)
small = TinyGPT().to(device)
small.load_state_dict({k: (v[0].float() * v[1]) if isinstance(v, tuple) else v for k, v in q.items()})
# Check that the shrunk model still writes stories
model = small; print(generate("Instruction: Write a short story about a cat.\nStory: "))
# Share it with the world (needs a free Hugging Face account and token):
# from huggingface_hub import HfApi; HfApi().upload_folder(folder_path=".", repo_id="your-name/tiny-story-gpt")